In [5]:
import pandas as pd
import re
import requests
import time


In [6]:
# List of cards that are allowed to have multiple copies in a Commander deck
SINGLETON_EXCEPTIONS = {
    "Shadowborn Apostle", "Persistent Petitioners", "Rat Colony", 
    "Dragon's Approach", "Slime Against Humanity", "Relentless Rats"
}

def get_scryfall_data(card_name):
    try:
        url = f"https://api.scryfall.com/cards/named?exact={requests.utils.quote(card_name)}"
        response = requests.get(url, timeout=10)
        return response.json() if response.status_code == 200 else None
    except Exception:
        return None

def parse_complex_mana(cost_str):
    if not cost_str:
        return {'w':0,'u':0,'b':0,'r':0,'g':0,'c':0,'x':0,'phi_opt':0,'hybrids':[]}
    symbols = re.findall(r'\{([^}]+)\}', cost_str)
    data = {'w':0,'u':0,'b':0,'r':0,'g':0,'c':0,'x':0,'phi_opt':0,'hybrids':[]}
    for s in [sym.upper() for sym in symbols]:
        if s in 'WUBRG': data[s.lower()] += 1
        elif s == 'C': data['c'] += 1
        elif s == 'X': data['x'] += 1
        elif s.isdigit(): data['c'] += int(s)
        elif '/P' in s:
            data[s.replace('/P', '').lower()] += 1
            data['phi_opt'] += 2
        elif '/' in s:
            data['hybrids'].append(s)
            first = s.split('/')[0].lower()
            if first in data: data[first] += 1
            elif first.isdigit(): data['c'] += int(first)
    return data

def process_commander_deck(input_file='moxfield_export.txt'):
    with open(input_file, 'r') as f:
        lines = [l.strip() for l in f if l.strip()]

    # Assuming the last line is the Commander based on typical Moxfield exports
    library_lines = lines[:-1]
    commander_line = lines[-1]
    
    deck_list = []
    cache = {}
    seen_names = set()
    duplicates = []

    print(f"--- 🔍 STARTING VERBOSE DATA GRAB ---")
    
    # 1. Process Library
    for line in library_lines:
        match = re.match(r'^(\d+)\s+(.*)$', line)
        if not match: continue
        qty, name = int(match.group(1)), match.group(2).strip()

        # Uniqueness Check
        if name in seen_names and name not in SINGLETON_EXCEPTIONS:
            duplicates.append(name)
        seen_names.add(name)

        if name not in cache:
            print(f"[FETCHING] {name}...")
            raw = get_scryfall_data(name)
            if raw:
                face = raw.get('card_faces', [raw])[0]
                p = parse_complex_mana(raw.get('mana_cost', face.get('mana_cost', '')))
                cache[name] = {
                    'name': raw.get('name'),
                    'type_line': raw.get('type_line'),
                    'cmc': int(raw.get('cmc', 0)),
                    'mana_cost': raw.get('mana_cost', face.get('mana_cost', '')),
                    'w': p['w'], 'u': p['u'], 'b': p['b'], 'r': p['r'], 'g': p['g'], 'c': p['c'],
                    'x_count': p['x'], 'phi_life_opt': p['phi_opt'], 'hybrid_opts': "|".join(p['hybrids']),
                    'effects_line': raw.get('oracle_text', face.get('oracle_text', '')).replace('\n', ' '),
                    'is_commander': False
                }
            time.sleep(0.12)

        if name in cache:
            for _ in range(qty):
                deck_list.append(cache[name].copy())

    # 2. Process Commander
    comm_match = re.match(r'^(\d+)\s+(.*)$', commander_line)
    if comm_match:
        comm_name = comm_match.group(2).strip()
        print(f"[COMMANDER] Fetching: {comm_name}...")
        raw_comm = get_scryfall_data(comm_name)
        if raw_comm:
            # Re-use cache or fetch
            p_c = parse_complex_mana(raw_comm.get('mana_cost', ''))
            comm_data = {
                'name': raw_comm.get('name'),
                'type_line': raw_comm.get('type_line'),
                'cmc': int(raw_comm.get('cmc', 0)),
                'mana_cost': raw_comm.get('mana_cost', ''),
                'w': p_c['w'], 'u': p_c['u'], 'b': p_c['b'], 'r': p_c['r'], 'g': p_c['g'], 'c': p_c['c'],
                'x_count': p_c['x'], 'phi_life_opt': p_c['phi_opt'], 'hybrid_opts': "|".join(p_c['hybrids']),
                'effects_line': raw_comm.get('oracle_text', '').replace('\n', ' '),
                'is_commander': True
            }
            deck_list.append(comm_data)

    # Final Validation
    print(f"\n--- 📊 DECK VALIDATION REPORT ---")
    print(f"Total Cards Found: {len(deck_list)}")
    
    if len(deck_list) != 100:
        print(f"⚠️ WARNING: Deck size is {len(deck_list)}, not 100.")
    
    if duplicates:
        print(f"⚠️ NON-UNIQUE CARDS DETECTED: {', '.join(set(duplicates))}")
    else:
        print("✅ Uniqueness Check Passed (Singleton Rule maintained).")

    print(f"👑 COMMANDER: {comm_name}")
    
    df = pd.DataFrame(deck_list)
    df.to_csv('mtg_enriched_data.csv', index=False)
    return df

In [7]:
df = process_commander_deck()

--- 🔍 STARTING VERBOSE DATA GRAB ---
[FETCHING] Adeline, Resplendent Cathar...
[FETCHING] Adriana, Captain of the Guard...
[FETCHING] Agadeem's Awakening...
[FETCHING] Akroma's Will...
[FETCHING] Anthem of Rakdos...
[FETCHING] Arcane Signet...
[FETCHING] Archon of Emeria...
[FETCHING] Aurelia, the Warleader...
[FETCHING] Aven Mindcensor...
[FETCHING] Battlefield Forge...
[FETCHING] Blood Crypt...
[FETCHING] Boros Signet...
[FETCHING] Breena, the Demagogue...
[FETCHING] Brimaz, King of Oreskos...
[FETCHING] Captain Lannery Storm...
[FETCHING] Castle Embereth...
[FETCHING] Caves of Koilos...
[FETCHING] City of Brass...
[FETCHING] Combat Calligrapher...
[FETCHING] Combat Celebrant...
[FETCHING] Command Tower...
[FETCHING] Curse of Opulence...
[FETCHING] Dark Ritual...
[FETCHING] Deafening Silence...
[FETCHING] Den of the Bugbear...
[FETCHING] Divine Visitation...
[FETCHING] Drakuseth, Maw of Flames...
[FETCHING] Drannith Magistrate...
[FETCHING] Emberwilde Captain...
[FETCHING] Esper Sent